# Z24 experiment runner

This notebook contains no model logic. Select one config, then run all cells. Attach the dataset containing `inputs.npy` and `labels.npy`; the runner locates it under `/kaggle/input`. Run each experiment in a fresh Kaggle session so TensorFlow and PyTorch do not retain each other's GPU memory.

In [ ]:
# Cell 1 - settings you may change
from pathlib import Path

REPO_URL = 'https://github.com/tranvanphuongdevdream-web/shm_ml.git'
BRANCH = 'master'
EXPERIMENT_ID = 'dcnn_002'  # dcnn_001 | dcnn_002 | tsai_001
OUTPUT_DIR = Path('/kaggle/working/results')
REPO_DIR = Path('/kaggle/working/shm_ml')

assert EXPERIMENT_ID in {'dcnn_001', 'dcnn_002', 'tsai_001'}


In [ ]:
# Cell 2 - clone once; later runs update without deleting the directory
import importlib
import os
import subprocess
import sys

git_environment = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
def run_git(arguments):
    return subprocess.run(
        arguments, check=True, timeout=180, env=git_environment,
    )

if (REPO_DIR / '.git').is_dir():
    print('Updating repository...', flush=True)
    run_git(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])
    run_git(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', '--depth', '1', 'origin', BRANCH])
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository')
else:
    print('Cloning repository...', flush=True)
    run_git(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('Repository ready:', REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
from src.data import z24_dataset
importlib.reload(z24_dataset)  # use code just pulled, not a cached module
INPUTS_PATH, LABELS_PATH = z24_dataset.resolve_data_source()
print('Dataset inputs:', INPUTS_PATH)
print('Dataset labels:', LABELS_PATH)


In [ ]:
# Cell 3 - one requirements file; keep Kaggle's CUDA-enabled PyTorch
from importlib.metadata import PackageNotFoundError, version

if EXPERIMENT_ID == 'tsai_001':
    requirements = [
        line.strip()
        for line in (REPO_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
        if line.strip() and not line.lstrip().startswith('#')
    ]
    tsai_specs = [spec for spec in requirements if spec.startswith('tsai==')]
    if len(tsai_specs) != 1:
        raise ValueError('requirements.txt must contain exactly one tsai== version')
    other_specs = [spec for spec in requirements if spec != tsai_specs[0]]
    try:
        torch_before = version('torch')
    except PackageNotFoundError:
        torch_before = None
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        *other_specs,
    ])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        '--no-deps', tsai_specs[0],
    ])
    assert version('torch') == torch_before, 'The Kaggle PyTorch build changed unexpectedly'
print('Dependencies ready for:', EXPERIMENT_ID)


In [ ]:
# Cell 4 - run in a child process so framework GPU memory is released on exit
existing_runs = {path for path in OUTPUT_DIR.glob(f'{EXPERIMENT_ID}_*') if path.is_dir()}
command = [
    sys.executable, '-u', 'run.py', 'train', '--experiment', EXPERIMENT_ID,
]
print('Running:', ' '.join(command))
subprocess.check_call(command, cwd=REPO_DIR)
new_runs = [
    path for path in OUTPUT_DIR.glob(f'{EXPERIMENT_ID}_*')
    if path.is_dir() and path not in existing_runs
]
if len(new_runs) != 1:
    raise RuntimeError(f'Expected one new result directory, found: {new_runs}')
RUN_DIR = new_runs[0]
print('Results:', RUN_DIR)
print('Download ZIP:', RUN_DIR.with_suffix('.zip'))


In [ ]:
# Cell 5 - metrics and training benchmark for this run
import json
import pandas as pd
from IPython.display import display

metrics = pd.read_csv(RUN_DIR / 'split_metrics.csv', index_col='split')
benchmark = json.loads((RUN_DIR / 'benchmark_summary.json').read_text(encoding='utf-8'))
print('Accuracy, macro precision, macro recall, and macro F1:')
display(metrics.style.format('{:.2%}'))

benchmark_rows = [
    ('Experiment', benchmark['pipeline']),
    ('Total training time (min)', round(benchmark['training_seconds_total'] / 60, 2)),
    ('Epochs completed', benchmark['epochs_completed']),
    ('Mean epoch time (s)', round(benchmark['mean_epoch_seconds'], 2)),
    ('Mean epoch time, excluding first (s)', round(benchmark['mean_epoch_seconds_excluding_first'], 2)),
    ('Train samples per second', round(benchmark['effective_train_samples_per_second'], 2)),
    ('Model parameters', benchmark['model_parameters']),
    ('Batch size', benchmark['batch_size']),
    ('GPU', ', '.join(benchmark['gpu_names']) or 'CPU'),
    ('Precision policy', benchmark['precision_policy']),
]
print('Training benchmark:')
display(pd.DataFrame(benchmark_rows, columns=['Metric', 'Value']).set_index('Metric'))


In [ ]:
# Cell 6 - learning curves, split comparison, and test confusion matrix
import matplotlib.pyplot as plt
from IPython.display import Image

history = pd.read_csv(RUN_DIR / 'history.csv')
if not history.empty:
    epoch = history['epoch'] if 'epoch' in history else range(1, len(history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for axis, title, columns in (
        (axes[0], 'Loss by epoch', ('loss', 'val_loss', 'train_loss', 'valid_loss')),
        (axes[1], 'Accuracy by epoch', ('accuracy', 'val_accuracy')),
    ):
        available = [column for column in columns if column in history]
        for column in available:
            axis.plot(epoch, history[column], label=column)
        if available:
            axis.set_title(title)
            axis.set_xlabel('Epoch')
            axis.grid(True, alpha=0.3)
            axis.legend()
        else:
            axis.set_visible(False)
    fig.tight_layout()
    plt.show()

axis = metrics[['accuracy', 'f1_macro']].plot.bar(figsize=(8, 4), rot=0)
axis.set_title('Performance by split')
axis.set_ylabel('Score')
axis.set_ylim(0, 1)
axis.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Test confusion matrix:')
display(Image(filename=str(RUN_DIR / 'test_confusion_matrix.png'), width=850))
